# Sửa ảnh host cho giống ảnh chụp thật (ADR-061)

Notebook chạy **trên Kaggle**, không chạy trên laptop. Nó bật ComfyUI không giao diện web và dùng **Qwen-Image-Edit-2511** để sửa
ảnh host đã chọn (`demo_host_02`, seed 606): giữ nguyên người, dáng ngồi, áo, bố cục và shop; chỉ làm da, mắt, tay, ánh sáng và kệ
hàng trông như ảnh chụp thật. Ảnh được chọn sẽ làm `start.png` cho InfiniteTalk (`make_infinitetalk_video.ipynb`).

**Quy tắc (CLAUDE.md, ADR-036, ADR-060):**
- Ảnh gốc là gương mặt AI, người không có thật. Không đưa ảnh người thật vào notebook, kể cả ảnh tham khảo.
- Không vẽ logo hay thương hiệu có thật.
- Mỗi nhân vật chỉ dùng cho một khách, một shop.
- Ảnh và `provenance_edit.json` **không đưa vào git**. Tải về laptop, lưu ở `assets/avatars/<CHARACTER_ID>/`.
- Môi luôn khép, không lộ răng: ảnh này là khung chờ của lip-sync.

**Đầu vào:** output của notebook Kaggle `ai-host-scene-demo-host-02` (gắn qua *Add Input → Notebook Output*). Notebook tự tìm file
`SOURCE_IMAGE_NAME` trong `/kaggle/input`.

**Cách chạy:** Accelerator = GPU T4, Internet = On, rồi Run All (hoặc Save & Run All). Kết quả ở `avatar_out/<CHARACTER_ID>/edit/`,
kèm `contact_sheet_edit.jpg` (cả ảnh) và `face_compare_edit.jpg` (mặt và tay ở độ phân giải gốc) để chọn.

Model: Qwen-Image-Edit-2511 bản GGUF Q4_K_M, LoRA Lightning 8 bước, đều Apache-2.0. Không dùng FLUX.1 Kontext dev (phi thương mại).

## 1. Cấu hình

Chỉ sửa ô này. Mọi giá trị được ghi vào `provenance_edit.json`.

In [ ]:
CHARACTER_ID = "demo_host_02"  # one character = one customer, one shop
CREATED_BY = "jintran0501"  # who runs this notebook; required in provenance
SOURCE_IMAGE_NAME = "scene_real_chest_up_seed606.png"  # chosen in run 2 of make_host_scene.ipynb
SOURCE_IMAGE_PATH = ""  # set on Colab or when the file is not under /kaggle/input
# When the input is not attached, fetch the file from this notebook's saved output with kagglehub.
SOURCE_NOTEBOOK = "jintran0501/ai-host-scene-demo-host-02"

# At 776x1352 the reference latent that TextEncodeQwenImageEditPlus builds (about 1 megapixel, multiple of 8) has
# exactly the output size, so the edit does not shift or zoom the picture. The aspect differs from 768x1344 by 0.4%.
EDIT_SIZE = (776, 1352)
OUTPUT_SIZE = (768, 1344)  # same as the source, so before/after line up pixel for pixel
STEPS = 8  # the Lightning LoRA is distilled for 8 steps at cfg 1
SHIFT = 3.1  # ModelSamplingAuraFlow value of the official ComfyUI 2511 template
SEEDS = [11, 22, 33]

# Repeated in every pass: an edit model drifts when an instruction leaves the rest of the picture open.
KEEP = (
    "Keep the same woman with exactly the same face, identity, facial proportions, hairstyle and low ponytail, the "
    "same plain navy blue polo shirt with orange trim, the same sitting pose, camera angle, framing and composition, "
    "and the same skincare shop behind her. Her lips stay closed and relaxed with no teeth showing, calm friendly "
    "expression. Do not redesign the character and do not add text or logos."
)
# Three short edits applied one after another; one instruction at a time keeps the layout stable.
PASSES = {
    "skin_eyes": (
        "Retouch only her skin and eyes so the picture looks like an unretouched photograph of a real person: natural "
        "skin texture with fine pores, slight natural unevenness and tiny imperfections, matte skin with no shine, "
        "fine hairs at the hairline, moist natural eyes with soft catchlights, detailed irises, individual eyelashes "
        "and eyebrow hairs. The skin must not look smooth, waxy, plastic or airbrushed."
    ),
    "hands": (
        "Fix only her hands resting on the desk: two natural relaxed hands, each with exactly five fingers, correct "
        "knuckles and finger joints, short natural nails, fingers loosely resting together without tangling or "
        "merging. Leave the rest of the picture unchanged."
    ),
    "light_bg": (
        "Make the picture look like a professional photograph taken with a full frame camera and a 50mm lens at f/2.8: "
        "soft natural key light from the front left with gentle shadow falloff on the face and neck, true to life "
        "colors, slight depth of field so the shelves behind her are a little softer than her face, realistic shelf "
        "products with small natural variations, reflections and imperfect spacing. Keep her face, hands and clothes "
        "unchanged."
    ),
}
# For comparison: everything in one instruction.
COMBINED = " ".join(PASSES.values())

EXTRA_SERVER_ARGS: list[str] = []
# Turing GPUs have no bf16, and Qwen-Image may overflow in fp16 (all-black image). Emulated bf16 is slower but safe.
FALLBACK_SERVER_ARGS = ["--bf16-unet"]

## 2. Môi trường, ComfyUI và node GGUF

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from datetime import UTC, datetime
from pathlib import Path

import torch  # preinstalled by Kaggle/Colab with the right CUDA build

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
if not (ON_KAGGLE or ON_COLAB):
    raise SystemExit("Notebook này chỉ chạy trên Kaggle hoặc Colab (ADR-036, ADR-061), không chạy trên laptop.")
PLATFORM = "kaggle" if ON_KAGGLE else "colab"

# Weights stay out of /kaggle/working: that folder is the saved output and is capped at 20 GB.
COMFY_DIR = Path("/tmp/ComfyUI") if ON_KAGGLE else Path("/content/ComfyUI")
OUT_ROOT = (Path("/kaggle/working") if ON_KAGGLE else Path("/content")) / "avatar_out"
OUT_DIR = OUT_ROOT / CHARACTER_ID
EDIT_DIR = OUT_DIR / "edit"
EDIT_DIR.mkdir(parents=True, exist_ok=True)

GGUF_NODE_REPO = "https://github.com/city96/ComfyUI-GGUF"
GGUF_NODE_COMMIT = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"  # main on 2026-11-17 (ADR-061)


def run(*cmd: str, cwd: Path | None = None) -> str:
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} lỗi:\n{result.stderr[-3000:]}")
    return result.stdout.strip()


def pip_requirements(path: Path) -> list[str]:
    # Keep the platform's CUDA build of torch: reinstalling it from PyPI can break the GPU runtime.
    skip = {"torch", "torchvision", "torchaudio"}
    reqs = []
    for line in path.read_text().splitlines():
        line = line.strip()
        name = line.split(";")[0].split("[")[0]
        for sep in "<>=!~ ":
            name = name.split(sep)[0]
        if line and not line.startswith("#") and name.lower() not in skip:
            reqs.append(line)
    return reqs


if not COMFY_DIR.exists():
    run("git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY_DIR))
COMFY_COMMIT = run("git", "rev-parse", "HEAD", cwd=COMFY_DIR)

GGUF_DIR = COMFY_DIR / "custom_nodes" / "ComfyUI-GGUF"
if not GGUF_DIR.exists():
    run("git", "clone", GGUF_NODE_REPO, str(GGUF_DIR))
run("git", "checkout", "-q", GGUF_NODE_COMMIT, cwd=GGUF_DIR)

run(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    *pip_requirements(COMFY_DIR / "requirements.txt"),
    *pip_requirements(GGUF_DIR / "requirements.txt"),
    "huggingface_hub",
)

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
free_gb = shutil.disk_usage("/tmp").free / 1e9
print(f"Nền tảng: {PLATFORM} · GPU: {GPU_NAME} · torch {torch.__version__} · ComfyUI {COMFY_COMMIT[:10]}")
print(f"Ổ đĩa /tmp còn {free_gb:.0f} GB (cần khoảng 25 GB cho model)")
if GPU_NAME == "none":
    raise SystemExit("Chưa bật GPU. Kaggle: Accelerator = GPU T4. Colab: Runtime type = T4 GPU.")

## 3. Ảnh gốc

In [ ]:
from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageStat

if SOURCE_IMAGE_PATH:
    source_path = Path(SOURCE_IMAGE_PATH)
else:
    matches = sorted(Path("/kaggle/input").rglob(SOURCE_IMAGE_NAME)) if ON_KAGGLE else []
    if not matches and ON_KAGGLE:
        # A notebook saved through the API may start without its inputs attached; kagglehub authenticates as the
        # notebook owner inside Kaggle and can download a private notebook's output directly.
        # A child process picks up an upgraded kagglehub without restarting the kernel.
        fetch = f"import kagglehub; print(kagglehub.notebook_output_download({SOURCE_NOTEBOOK!r}))"
        try:
            run(sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub")
            fetched = Path(run(sys.executable, "-c", fetch).splitlines()[-1])
            matches = sorted(fetched.rglob(SOURCE_IMAGE_NAME))
            print(f"Lấy ảnh gốc từ output của {SOURCE_NOTEBOOK}: {fetched}")
        except (RuntimeError, IndexError) as err:
            print(f"kagglehub không lấy được output của {SOURCE_NOTEBOOK}: {err}")
    if not matches:
        # Show what is mounted: a wrong input name is easier to spot than a missing one.
        mounted = sorted(str(p) for p in Path("/kaggle/input").rglob("*"))[:40] if ON_KAGGLE else []
        raise SystemExit(
            f"Không thấy {SOURCE_IMAGE_NAME} trong /kaggle/input (đang có: {mounted}). Thêm output của notebook "
            f"{SOURCE_NOTEBOOK} (Add Input → Notebook Output) hoặc đặt SOURCE_IMAGE_PATH."
        )
    source_path = matches[0]
SOURCE_SHA256 = hashlib.sha256(source_path.read_bytes()).hexdigest()
source_image = PILImage.open(source_path).convert("RGB")
print(f"Ảnh gốc: {source_path} · {source_image.size[0]}x{source_image.size[1]} · sha256 {SOURCE_SHA256[:12]}")
if source_image.size != OUTPUT_SIZE:
    raise SystemExit(f"Ảnh gốc phải là {OUTPUT_SIZE}, đang là {source_image.size}.")

# LoadImage only reads from ComfyUI's input folder. Every pass works at EDIT_SIZE; only saved copies go back to
# OUTPUT_SIZE, so a chain of passes is resampled once instead of after every pass.
COMFY_INPUT = COMFY_DIR / "input"
COMFY_INPUT.mkdir(exist_ok=True)
SOURCE_INPUT = "source_edit_size.png"
source_image.resize(EDIT_SIZE, PILImage.LANCZOS).save(COMFY_INPUT / SOURCE_INPUT)
shutil.copy(source_path, EDIT_DIR / "source.png")

## 4. Tải model (chỉ trong máy Kaggle, không về laptop)

Khoảng 23 GB. Revision ghim theo ngày kiểm 2026-11-17 (ADR-061).

In [ ]:
from huggingface_hub import hf_hub_download

MODELS = {
    "edit": {
        "repo": "unsloth/Qwen-Image-Edit-2511-GGUF",
        "file": "qwen-image-edit-2511-Q4_K_M.gguf",
        "revision": "52838251db242ddd8e8d0118de36c71006d7c397",
        "dir": "diffusion_models",
        "license": "Apache-2.0 (Qwen-Image-Edit-2511, base Qwen/Qwen-Image-Edit-2511 @ 6f3ccc0b56)",
    },
    "lightning_lora": {
        "repo": "lightx2v/Qwen-Image-Edit-2511-Lightning",
        "file": "Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors",
        "revision": "107d238d68b8387e32843d0b4a2e24ceb0d3ad86",
        "dir": "loras",
        "license": "Apache-2.0",
    },
    "text_encoder": {
        "repo": "Comfy-Org/Qwen-Image_ComfyUI",
        "file": "split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors",
        "revision": "1f12b17be14c89b026c51a91d67c32f84bb047bc",
        "dir": "text_encoders",
        "license": "Apache-2.0 (Qwen2.5-VL-7B)",
    },
    "vae": {
        "repo": "Comfy-Org/Qwen-Image_ComfyUI",
        "file": "split_files/vae/qwen_image_vae.safetensors",
        "revision": "1f12b17be14c89b026c51a91d67c32f84bb047bc",
        "dir": "vae",
        "license": "Apache-2.0 (Qwen-Image)",
    },
}

for key, model in MODELS.items():
    started = time.monotonic()
    cached = Path(hf_hub_download(model["repo"], model["file"], revision=model["revision"]))
    target = COMFY_DIR / "models" / model["dir"] / cached.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.symlink_to(cached)
    model["name"] = cached.name  # ComfyUI loaders take the bare file name
    size_gb = cached.stat().st_size / 1e9
    print(f"{key}: {cached.name} @ {model['revision'][:10]} · {size_gb:.2f} GB ({time.monotonic() - started:.0f} s)")

## 5. Bật ComfyUI (chỉ nghe trong máy, không mở giao diện web)

In [ ]:
SERVER = "http://127.0.0.1:8188"
server: subprocess.Popen | None = None
server_args: list[str] = []


def log_tail() -> str:
    return (OUT_ROOT / "comfyui.log").read_text(errors="replace")[-3000:]


def api(path: str, payload: dict | None = None) -> dict:
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(SERVER + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read()
    return json.loads(body) if body else {}


def start_server(args: list[str]) -> None:
    """(Re)start ComfyUI with the given extra arguments and wait until it answers."""
    global server, server_args
    if server is not None:
        server.terminate()
        server.wait(timeout=60)
    server_args = list(args)
    # Appending keeps the log of an earlier start when the fp16 fallback restarts the server.
    log = open(OUT_ROOT / "comfyui.log", "a")
    server = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--preview-method", "none", *args],
        cwd=COMFY_DIR,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    for _ in range(120):
        if server.poll() is not None:
            raise RuntimeError("ComfyUI dừng khi khởi động:\n" + log_tail())
        try:
            api("/system_stats")
            break
        except OSError:
            time.sleep(2)
    else:
        raise RuntimeError("ComfyUI không lên sau 4 phút:\n" + log_tail())
    nodes = api("/object_info")
    missing = [n for n in ("UnetLoaderGGUF", "TextEncodeQwenImageEditPlus", "CFGNorm") if n not in nodes]
    if missing:
        raise RuntimeError(f"ComfyUI thiếu node {missing}:\n" + log_tail())
    print(f"ComfyUI đã chạy với {args or 'tham số mặc định'}.")


def run_workflow(workflow: dict, timeout_s: float) -> dict:
    """Queue one API-format workflow and wait for its outputs."""
    try:
        prompt_id = api("/prompt", {"prompt": workflow})["prompt_id"]
    except urllib.error.HTTPError as err:
        # ComfyUI explains a rejected workflow (missing node, wrong input) in the body.
        raise RuntimeError(err.read().decode(errors="replace")[:3000]) from err
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        entry = api(f"/history/{prompt_id}").get(prompt_id)
        if entry is not None:
            if entry.get("status", {}).get("status_str") == "error":
                raise RuntimeError(json.dumps(entry["status"], ensure_ascii=False)[:3000])
            return entry["outputs"]
        if server.poll() is not None:
            raise RuntimeError("ComfyUI đã dừng giữa chừng:\n" + log_tail())
        time.sleep(5)
    raise TimeoutError(f"Quá {timeout_s:.0f} s mà workflow chưa xong")


def output_images(outputs: dict) -> list[Path]:
    files = []
    for node_output in outputs.values():
        for image in node_output.get("images", []):
            files.append(COMFY_DIR / "output" / image["subfolder"] / image["filename"])
    return files


start_server(EXTRA_SERVER_ARGS)

## 6. Sửa ảnh

Đồ thị node theo template chính thức `image_qwen_image_edit_2511.json` của ComfyUI, thay bộ tải model bằng bản GGUF.

- **Chuỗi 3 lần sửa** cho mỗi seed: da và mắt → tay → ánh sáng và nền. Lần sau sửa tiếp trên ảnh của lần trước.
- **Một lần gộp** cả 3 ý trên ảnh gốc, để so.

Ảnh đầu tiên được kiểm độ sáng: nếu gần như đen (fp16 tràn số trên T4) thì bật lại ComfyUI với `FALLBACK_SERVER_ARGS` và làm lại.

In [ ]:
def edit_workflow(input_name: str, instruction: str, seed: int, prefix: str) -> dict:
    width, height = EDIT_SIZE
    return {
        "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": MODELS["edit"]["name"]}},
        "2": {
            "class_type": "CLIPLoader",
            "inputs": {"clip_name": MODELS["text_encoder"]["name"], "type": "qwen_image", "device": "default"},
        },
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": MODELS["vae"]["name"]}},
        "4": {"class_type": "ModelSamplingAuraFlow", "inputs": {"model": ["1", 0], "shift": SHIFT}},
        "5": {"class_type": "CFGNorm", "inputs": {"model": ["4", 0], "strength": 1.0}},
        "6": {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {"model": ["5", 0], "lora_name": MODELS["lightning_lora"]["name"], "strength_model": 1.0},
        },
        "7": {"class_type": "LoadImage", "inputs": {"image": input_name}},
        # The input is already EDIT_SIZE; the node only guards against a wrong size.
        "8": {
            "class_type": "ImageScale",
            "inputs": {
                "image": ["7", 0],
                "upscale_method": "lanczos",
                "width": width,
                "height": height,
                "crop": "disabled",
            },
        },
        "9": {
            "class_type": "TextEncodeQwenImageEditPlus",
            "inputs": {"clip": ["2", 0], "prompt": instruction, "vae": ["3", 0], "image1": ["8", 0]},
        },
        # cfg 1 ignores the negative branch; KSampler still needs one, built like the template's empty prompt.
        "10": {
            "class_type": "TextEncodeQwenImageEditPlus",
            "inputs": {"clip": ["2", 0], "prompt": "", "vae": ["3", 0], "image1": ["8", 0]},
        },
        "11": {
            "class_type": "FluxKontextMultiReferenceLatentMethod",
            "inputs": {"conditioning": ["9", 0], "reference_latents_method": "index_timestep_zero"},
        },
        "12": {
            "class_type": "FluxKontextMultiReferenceLatentMethod",
            "inputs": {"conditioning": ["10", 0], "reference_latents_method": "index_timestep_zero"},
        },
        "13": {"class_type": "VAEEncode", "inputs": {"pixels": ["8", 0], "vae": ["3", 0]}},
        "14": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["6", 0],
                "positive": ["11", 0],
                "negative": ["12", 0],
                "latent_image": ["13", 0],
                "seed": seed,
                "steps": STEPS,
                "cfg": 1.0,
                "sampler_name": "euler",
                "scheduler": "simple",
                "denoise": 1.0,
            },
        },
        "15": {"class_type": "VAEDecode", "inputs": {"samples": ["14", 0], "vae": ["3", 0]}},
        "16": {"class_type": "SaveImage", "inputs": {"images": ["15", 0], "filename_prefix": prefix}},
    }


def mean_luma(path: Path) -> float:
    return round(ImageStat.Stat(PILImage.open(path).convert("L")).mean[0], 1)


SOURCE_LUMA = mean_luma(EDIT_DIR / "source.png")
fallback_used = False


def edit_once(input_name: str, instruction: str, seed: int, name: str) -> dict:
    """Run one edit, keep the EDIT_SIZE result as a ComfyUI input for the next pass, save an OUTPUT_SIZE copy."""
    global fallback_used
    path = EDIT_DIR / f"{name}.png"
    started = time.monotonic()
    workflow = edit_workflow(input_name, f"{instruction} {KEEP}", seed, f"{CHARACTER_ID}_{name}")
    [image] = output_images(run_workflow(workflow, timeout_s=2400))
    if mean_luma(image) < 5 and not fallback_used:
        # fp16 overflow shows up as an all-black image; retry once with emulated bf16.
        print(f"{name}: ảnh gần như đen, bật lại ComfyUI với {FALLBACK_SERVER_ARGS} rồi làm lại")
        fallback_used = True
        start_server(EXTRA_SERVER_ARGS + FALLBACK_SERVER_ARGS)
        [image] = output_images(run_workflow(workflow, timeout_s=3600))
    next_input = f"{name}_edit_size.png"
    shutil.copy(image, COMFY_INPUT / next_input)
    PILImage.open(image).convert("RGB").resize(OUTPUT_SIZE, PILImage.LANCZOS).save(path)
    luma = mean_luma(path)
    seconds = round(time.monotonic() - started, 1)
    print(f"{path.name}: {seconds} s, độ sáng trung bình {luma} (ảnh gốc {SOURCE_LUMA})")
    if luma < 5:
        raise RuntimeError(f"{name}: ảnh vẫn đen sau khi bật {server_args}. Xem {OUT_ROOT / 'comfyui.log'}.")
    return {
        "file": f"edit/{path.name}",
        "next_input": next_input,
        "seed": seed,
        "seconds": seconds,
        "mean_luma": luma,
        "server_args": server_args,
    }


chains: dict[int, list[dict]] = {}
combined: dict[int, dict] = {}
for seed in SEEDS:
    current = SOURCE_INPUT
    chains[seed] = []
    for step, (pass_name, instruction) in enumerate(PASSES.items(), start=1):
        result = edit_once(current, instruction, seed, f"seed{seed}_{step}_{pass_name}")
        chains[seed].append({"pass": pass_name, "input": current, **result})
        current = result["next_input"]
    combined[seed] = edit_once(SOURCE_INPUT, COMBINED, seed, f"seed{seed}_combined")

## 7. Ảnh so sánh

- `contact_sheet_edit.jpg`: mỗi hàng một seed: ảnh gốc, sau lần 1, 2, 3, rồi bản gộp.
- `face_compare_edit.jpg`: cắt mặt và tay ở độ phân giải gốc, vì ảnh thu nhỏ không thấy khác biệt về da và ngón tay. Hàng đầu là
  ảnh gốc, mỗi hàng sau là một seed (ảnh cuối chuỗi, rồi bản gộp).

In [ ]:
THUMB = (384, 672)
rows = [
    [EDIT_DIR / "source.png"] + [OUT_DIR / c["file"] for c in chains[seed]] + [OUT_DIR / combined[seed]["file"]]
    for seed in SEEDS
]
columns = max(len(row) for row in rows)
sheet = PILImage.new("RGB", (THUMB[0] * columns, THUMB[1] * len(rows)), "white")
for row_index, row in enumerate(rows):
    for column, file in enumerate(row):
        sheet.paste(PILImage.open(file).convert("RGB").resize(THUMB), (column * THUMB[0], row_index * THUMB[1]))
sheet.save(OUT_DIR / "contact_sheet_edit.jpg", quality=88)

# Fractions of width/height in the chest_up framing of seed 606.
FACE_BOX = (0.30, 0.22, 0.70, 0.52)
HANDS_BOX = (0.25, 0.66, 0.80, 0.88)
CROP = (420, 540)


def crop(file: Path, box: tuple[float, float, float, float]) -> PILImage.Image:
    image = PILImage.open(file).convert("RGB")
    w, h = image.size
    region = image.crop((int(box[0] * w), int(box[1] * h), int(box[2] * w), int(box[3] * h)))
    return region.resize(CROP)


compare_rows = [[EDIT_DIR / "source.png"]] + [
    [OUT_DIR / chains[seed][-1]["file"], OUT_DIR / combined[seed]["file"]] for seed in SEEDS
]
faces = PILImage.new("RGB", (CROP[0] * 4, CROP[1] * len(compare_rows)), "white")
for row_index, row in enumerate(compare_rows):
    for column, file in enumerate(row):
        faces.paste(crop(file, FACE_BOX), (column * 2 * CROP[0], row_index * CROP[1]))
        faces.paste(crop(file, HANDS_BOX), ((column * 2 + 1) * CROP[0], row_index * CROP[1]))
faces.save(OUT_DIR / "face_compare_edit.jpg", quality=92)
display(Image(filename=str(OUT_DIR / "contact_sheet_edit.jpg"), width=1000))
display(Image(filename=str(OUT_DIR / "face_compare_edit.jpg"), width=900))

## 8. Hồ sơ nguồn gốc

In [ ]:
provenance = {
    "character_id": CHARACTER_ID,
    "created_by": CREATED_BY,
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "kind": "ai_generated_fictional_face_edit",
    "adr": "ADR-061",
    "platform": PLATFORM,
    "platform_terms_note": "ADR-036: dùng cho giai đoạn demo; đọc lại điều khoản trước khi khách trả tiền",
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "comfyui_commit": COMFY_COMMIT,
    "custom_nodes": {"ComfyUI-GGUF": {"repo": GGUF_NODE_REPO, "commit": GGUF_NODE_COMMIT, "license": "Apache-2.0"}},
    "models": MODELS,
    "source": {"name": SOURCE_IMAGE_NAME, "path": str(source_path), "sha256": SOURCE_SHA256},
    "settings": {
        "edit_size": list(EDIT_SIZE),
        "output_size": list(OUTPUT_SIZE),
        "steps": STEPS,
        "shift": SHIFT,
        "sampler": "euler/simple, cfg 1, denoise 1, CFGNorm 1, index_timestep_zero",
        "keep": KEEP,
        "passes": PASSES,
        "fallback_used": fallback_used,
    },
    "chains": {str(seed): steps for seed, steps in chains.items()},
    "combined": {str(seed): result for seed, result in combined.items()},
}
(OUT_DIR / "provenance_edit.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
server.terminate()
print("Xong:", sorted(str(p.relative_to(OUT_DIR)) for p in OUT_DIR.rglob("*") if p.is_file()))